# Lab M.2 &mdash; Ask Your Traces

**Day 3 &middot; MCP &middot; about 25 min &middot; in your sandbox**

### What you'll do
- Add a second MCP server, Langfuse, to the agent from Lab M.1
- Ask your observability data questions in plain English
- Find your slowest step, how often each step runs, and which tools get called
- Leave with a prompt library you can reuse

> **How this lab works.** Run each cell in order with **Shift + Enter** and read what it prints. There is nothing to fill in and nothing is graded. Some steps print a command for you to paste into a JupyterLab terminal (**File &rarr; New &rarr; Terminal**).

## The use case

When you instrument agents with Langfuse, every run lands as a trace. Then the awkward part
starts. *Which step is slow? Is `retrieve` firing twice? Did anything error overnight? Is that tool
I shipped last week actually being called?*

Every one of those is answerable from the data you already have &mdash; and normally costs somebody
an export, a query, or twenty minutes of clicking. Langfuse publishes its API as an **MCP server**,
so your agent can answer them instead, in a sentence.

You are not building anything new here. You are adding one entry to the config you already wrote in
Lab M.1, and then asking questions.

In [ ]:
# ------------------------------------------------------------ Preflight: run me first
import os, json, time, base64, textwrap, subprocess, re, socket, urllib.request, urllib.error

MODEL  = os.environ.get("LAB_LLM_MODEL", "")
HOST   = os.environ.get("LANGFUSE_HOST", "")
PK     = os.environ.get("LANGFUSE_PUBLIC_KEY", "")
SK     = os.environ.get("LANGFUSE_SECRET_KEY", "")
MINE   = os.environ.get("LANGFUSE_TRACING_ENVIRONMENT", "")
LABDIR = os.path.expanduser("~/work/mcplab")        # the SAME folder as Lab M.1
CONFIG_PATH = os.path.join(LABDIR, "opencode.json")

MCP_URL = HOST.rstrip("/") + "/api/public/mcp" if HOST else ""
AUTH    = base64.b64encode(f"{PK}:{SK}".encode()).decode() if (PK and SK) else ""
if AUTH:
    os.environ["LANGFUSE_MCP_AUTH"] = AUTH     # for this notebook's own opencode calls only
LF_EXPORT = 'export LANGFUSE_MCP_AUTH=$(printf \'%s:%s\' "$LANGFUSE_PUBLIC_KEY" "$LANGFUSE_SECRET_KEY" | base64 -w0)'

def ready() -> bool:
    return bool(HOST and PK and SK)

print("lab M.1 config :", CONFIG_PATH, "-", "found" if os.path.exists(CONFIG_PATH) else "NOT FOUND")
if ready():
    print("langfuse       :", MCP_URL)
    print("your traces    :", MINE or "(environment tag not set)")
    os.makedirs(LABDIR, exist_ok=True)
else:
    print("\nNot configured. This lab reads three variables the sandbox already sets:")
    for n in ("LANGFUSE_HOST", "LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY"):
        print(f"  {n:22} {'set' if os.environ.get(n) else 'MISSING'}")
    print("Every cell below skips cleanly until they are set.")

## Step 1 &mdash; Add a second server to the agent you already have

Lab M.1 left an `opencode.json` in this folder with one MCP server in it. We are going to **add**
to it, not replace it &mdash; so the same agent ends up holding both Jira and Langfuse.

That is worth noticing on its own: an agent's capabilities are a list you extend, and each entry is
a separate grant with its own credential.

As in Lab M.1, the key stays out of the file. The header says `{env:LANGFUSE_MCP_AUTH}`, and the
next steps put that variable in your environment.

In [ ]:
def load_config() -> dict:
    """Read Lab M.1's config, or start a fresh one if you skipped that lab."""
    if os.path.exists(CONFIG_PATH):
        return json.load(open(CONFIG_PATH))
    return {
        "$schema": "https://opencode.ai/config.json",
        "provider": {"lab": {
            "npm": "@ai-sdk/openai-compatible", "name": "Lab model",
            "options": {"baseURL": "{env:LAB_LLM_BASE_URL}", "apiKey": "{env:OPENAI_API_KEY}"},
            "models": {MODEL: {"name": "Sandbox model"}}}},
        "mcp": {},
    }


if ready():
    cfg = load_config()
    before = sorted(cfg.get("mcp", {}))

    # ---- the whole integration: one more entry in the mcp block --------------
    cfg.setdefault("mcp", {})["langfuse"] = {
        "type": "remote",
        "url": MCP_URL,
        "enabled": True,
        # The key pair IS the project scope -- Langfuse works out which project to
        # answer for from these credentials. As with Jira, the file only names the
        # variable; the key itself stays in the environment.
        "headers": {"Authorization": "Basic {env:LANGFUSE_MCP_AUTH}"},
    }
    # Sensible default: let it read freely, never let it delete.
    cfg.setdefault("permission", {}).update({"langfuse_delete*": "deny"})

    with open(CONFIG_PATH, "w") as fh:
        json.dump(cfg, fh, indent=2)

    print("servers before :", before or "(none - you skipped Lab M.1, that is fine)")
    print("servers after  :", sorted(cfg["mcp"]))
    print("\nwrote", CONFIG_PATH)
else:
    print("skipped - see the preflight cell")

## Step 2 &mdash; Confirm both are live

In [ ]:
def oc(*args, timeout=180):
    p = subprocess.run(["opencode", *args], cwd=LABDIR, capture_output=True, text=True, timeout=timeout)
    return re.sub(r"\x1b\[[0-9;?]*[a-zA-Z]", "", (p.stdout or "") + (p.stderr or ""))

if ready():
    print(oc("mcp", "list"))
else:
    print("skipped - see the preflight cell")

## Step 3 &mdash; Ask it something you would otherwise have queried

In a terminal (**File &rarr; New &rarr; Terminal**), as in Lab M.1. Paste the `export` line first:
it puts the Langfuse key in that terminal's environment, where OpenCode reads it. Start with the
question below, then work through the library.

In [ ]:
FIRST = ("Call getMetricsSchema first. Then show me average and maximum latency by observation "
         "type and by name for the last 30 days, as a table sorted by average latency.")

if ready():
    print("Open File > New > Terminal, then paste:\n")
    print(LF_EXPORT)
    print(f"cd {LABDIR} && \\\n  opencode run --model lab/{MODEL} \\\n    \"{FIRST}\"")
else:
    print("skipped - see the preflight cell")

That table &mdash; every step you run, ranked by how slow it is &mdash; is the thing teams build a
dashboard for. You asked for it in one sentence, and the agent worked out the query.

Watch what it does when it gets something wrong, too. Langfuse rejects a bad dimension name with a
message listing the valid ones, and the agent simply tries again with the right one. **Good tool
errors are what make an agent recoverable** &mdash; and the reason your own tools should return errors, not raise them.

## The prompt library

This is what to take away. Every question below is one a team normally answers with an export, a
query, or twenty minutes of clicking. All of them were run against this project before being
written down.

**Three things that make them work:**

1. **Begin with *&ldquo;call getMetricsSchema first&rdquo;***. Otherwise the agent guesses a
   dimension name, gets rejected and retries. It recovers, but it costs turns.
2. **You share this project with the whole room.** Everyone's traces land here and only the
   `environment` tag separates them &mdash; yours is printed by the preflight cell above, and is
   also in `$LANGFUSE_TRACING_ENVIRONMENT`. Add *&ldquo;filter to environment = &lt;yours&gt;&rdquo;*
   to see only your own work.
3. ⚠️ **Cost reads zero, and that is true rather than broken.** `totalCost` is null on every
   observation because the sandbox model has no priced entry in Langfuse. Use the latency and
   structure questions; they have real data.

### Where the time goes

```
Call getMetricsSchema first. Then list the 10 slowest observations in the last 30 days with
their name, type and latency. What do the slow ones have in common?
```
*Finds your bottleneck. Generations and spans differ by three orders of magnitude.*

```
Compare average latency for the last 24 hours against the 24 hours before it.
Has anything regressed?
```
*A regression check without a dashboard. Run it each morning of a delivery.*

### What your agents are actually doing

```
Call getMetricsSchema first. Then break observations down by name for the last 30 days.
Which steps run most often, and does the ratio between them look right?
```
*How you notice `retrieve` firing three times when it should fire once. Nobody spots that
by reading traces.*

```
Which tools are being called, and how often? Use the calledToolNames dimension.
```
*The question that reveals a tool you shipped and nothing ever selects &mdash; which is a
description problem: rewrite the description.*

```
Are there observations with level ERROR or WARNING in the last 30 days? Show the most recent
five and summarise what they have in common.
```
*Triage without opening five traces by hand.*

```
Find the slowest observation in the last 30 days, fetch it in full, and explain in three
sentences what it was doing.
```
*Three tools off one sentence: metrics &rarr; list &rarr; fetch. This is the one that feels
like having an analyst.*

### Just yours, and housekeeping

```
Filter everything to environment = <your LANGFUSE_TRACING_ENVIRONMENT>. How many
observations are mine, what types are they, and which was slowest?
```
*The one you will use most, once the project is full of everyone's runs.*

```
List the prompts in this project with their labels and versions, and tell me which have no
production label.
```
*Prompt hygiene, which otherwise nobody audits until something breaks.*

### One that fails, informatively

```
What did this project cost last week, broken down by model?
```
*Returns zeros, for the instrumentation reason above. A good agent tells you the data is not
there and why; a weaker one invents a number. Worth finding out which you have &mdash; and it is
a fair reminder that **your observability is only ever as good as your instrumentation.**

## One safety default, and then you are done

Step 1 quietly added this alongside the server:

```json
"permission": { "langfuse_delete*": "deny" }
```

Langfuse publish their whole API, so the server offers plenty that deletes &mdash; dashboards,
datasets, evaluators, models. Reading is what you came for; deleting is not.

Try it and read the reply carefully:

```
Delete every dashboard in this project.
```

The agent does not say *&ldquo;I am not allowed.&rdquo;* It says the tool **does not exist**. `deny`
withholds the tool rather than policing the call, so there is nothing for the model to be argued
out of. A capability never offered beats a capability told not to use.

## What this actually bought you

One entry in a config file, and the questions at the top of this lab stopped needing a person.

| the question | what it used to cost |
|---|---|
| which step is slowest | a dashboard, or sorting traces by hand |
| is `retrieve` firing twice | reading traces one at a time |
| did anything error overnight | someone remembering to look |
| is that new tool ever called | parsing trace payloads |
| what did my own run do | filtering a shared project by hand |

None of that is new capability &mdash; the API could always answer it. What changed is that the
distance between having the question and having the answer is now one sentence, which is the
difference between a check you *could* run and one you actually do.

## Your turn

- Run the library against your own environment tag and see how thin it is today. Come back after
  the production labs and run it again &mdash; same prompts, real data.
- Take one answer and verify it in the Langfuse UI. Trust the agent only after that.
- Add a third server to the same config. Notice that nothing about the agent had to change.